# Проверка разметки по справочнику Авито

Сверяем новые версии с исходными данными и предыдущими версиями. Затем сравниваем одинаковую модель на одном разбиении. Метки исходного JSONL подставляем в те же очищенные реальные объявления, чтобы измерить вклад разметки.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'data/processed/train_guide_v1').is_dir())
sys.path.insert(0, str(root / 'research/experiments'))
sys.path.insert(0, str(root / 'dataset_review'))
from bench import canonical, digest, gold_overlap_key, key, read, baseline, metrics
from build_guide_v1 import raw_rows, raw_comparison, binary_labels, labels_for, sha256, load_hard_test

train_dir = root / 'data/processed/train_guide_v1'
test_dir = root / 'data/test/test_guide_v1'
train_manifest, test_manifest = [json.loads((folder / 'manifest.json').read_text())
                                 for folder in (train_dir, test_dir)]
for manifest in (train_manifest, test_manifest):
    assert sha256(root / 'dataset_review/build_guide_v1.py') == manifest['builder_sha256']
    for field in ('guide', 'review', 'label_changes'):
        info = manifest[field]
        assert sha256(root / info['path']) == info['sha256'], info['path']
    infos = list(manifest['sources'].values()) + ([manifest['dataset']] if 'dataset' in manifest
                                               else list(manifest['datasets'].values()))
    for info in infos:
        assert sha256(root / info['path']) == info['sha256'], info['path']

old_train = pq.read_table(root / train_manifest['sources']['train']['path']).to_pylist()
new_train = pq.read_table(train_dir / 'train.parquet').to_pylist()
assert len(old_train) == len(new_train) == train_manifest['dataset']['records']
assert len({row['item_id'] for row in new_train}) == len(new_train)
assert len({key(row['description']) for row in new_train}) == len(new_train)
train_changes = pd.read_csv(train_dir / 'label_changes.csv')
assert len(train_changes) == train_manifest['label_changes']['records']
changes_idx = {(row.row_id, row.mcId): (row.old, row.new) for row in train_changes.itertuples()}
actual_changes = {}
for old, new in zip(old_train, new_train, strict=True):
    assert {k: v for k, v in old.items() if k != 'found_microcategories'} == {
           k: v for k, v in new.items() if k != 'found_microcategories'}
    before, after = set(labels_for(old)), set(labels_for(new))
    for cat in before ^ after:
        actual_changes[(new['item_id'], cat)] = (int(cat in before), int(cat in after))
assert actual_changes == changes_idx
raw_info = train_manifest['sources']['raw_train']
raw = raw_rows(raw_info, new_train)
assert train_manifest['raw_comparison'] == {'parent_train': raw_comparison(old_train, raw),
                                           'guide_train': raw_comparison(new_train, raw)}
display(train_changes.groupby(['origin', 'mcId']).size().unstack(fill_value=0))
display(pd.DataFrame(train_manifest['raw_comparison']).loc[['rows', 'same_label_sets', 'same_description']])
print('Все строки и поля train, кроме сверенных меток, совпадают с предыдущей версией.')

mcId,102,103,104,105,106
origin,,,,,
real,1,1,2,1,35
synthetic,0,0,0,0,51


,parent_train,guide_train
rows,4214,4214
same_label_sets,3474,3445
same_description,921,921


Все строки и поля train, кроме сверенных меток, совпадают с предыдущей версией.


In [2]:
old_gold = pq.read_table(root / test_manifest['sources']['gold']['path']).to_pylist()
new_gold = pq.read_table(test_dir / 'test_gold.parquet').to_pylist()
old_stress = pq.read_table(root / test_manifest['sources']['stress']['path']).to_pylist()
new_stress = pq.read_table(test_dir / 'stress_suite.parquet').to_pylist()
old_hard = load_hard_test(root / test_manifest['sources']['hard']['path'])
new_hard = read(test_dir / 'hard.jsonl')
assert new_stress == read(test_dir / 'stress_suite.jsonl')
test_changes = pd.read_csv(test_dir / 'label_changes.csv', dtype={'row_id': str})
for name, before, after in [('gold', old_gold, new_gold), ('stress', old_stress, new_stress),
                            ('hard', old_hard, new_hard)]:
    assert len(before) == len(after) == test_manifest['sources'][name]['records']
    expected = {(r.row_id, r.mcId): (r.old, r.new)
                for r in test_changes.loc[test_changes.dataset == name].itertuples()}
    observed = {}
    ignored = {f'y_{c}' for c in range(102, 107)} | {'labels', 'labels_strict', 'labels_seo', 'note'}
    for old, new in zip(before, after, strict=True):
        assert {k: v for k, v in old.items() if k not in ignored} == {
               k: v for k, v in new.items() if k not in ignored}
        old_labels = set(binary_labels(old) if name == 'gold' else old['labels'])
        new_labels = set(binary_labels(new) if name == 'gold' else new['labels'])
        for cat in old_labels ^ new_labels:
            observed[(str(new['row_id']), cat)] = (int(cat in old_labels), int(cat in new_labels))
        if name == 'stress':
            assert new['labels'] == binary_labels(new)
            assert all(new['mcId'] not in new[field] for field in ('labels', 'labels_strict', 'labels_seo'))
    assert observed == expected
gold_by_id = {r['row_id']: binary_labels(r) for r in new_gold}
for row in new_stress:
    if row['origin_split'] == 'test_gold':
        assert row['labels'] == gold_by_id[int(row['note'].split('row_id=')[1])]

normalize = gold_overlap_key()
fit_keys = {key(row['description']) for row in new_train}
fit_normalized = {normalize(row['description']) for row in new_train} - {''}
public = pq.read_table(root / test_manifest['sources']['public']['path'],
                       columns=['description']).to_pylist()
for rows in (new_gold, public):
    assert not any(key(row['description']) in fit_keys or normalize(row['description']) in fit_normalized
                   for row in rows)
display(test_changes.groupby(['dataset', 'mcId']).size().unstack(fill_value=0))
display(test_changes.query("dataset == 'gold'")[['row_id', 'mcId', 'old', 'new', 'public_labels', 'quote']])
print('Тексты и ID тестов совпадают. Пересечений train с gold и публичным тестом после нормализации нет.')

mcId,102,103,104,105,106
dataset,,,,,
gold,4,1,2,2,4
hard,0,0,0,0,2
stress,4,1,2,2,19


,row_id,mcId,old,new,public_labels,quote
0,0,104,1,0,"{""0"": 0, ""28"": 0, ""54"": 1, ""70"": 1, ""135"": 1, ""167"": 1, ""207"": 1}","сантехнические работы, замена труб"
1,3,102,1,0,"{""3"": 1}",штукатурные работы
2,3,103,1,0,"{""3"": 1}",укладка плитки
3,3,104,1,0,"{""3"": 1}",сантехнические работы
4,3,105,1,0,"{""3"": 1}",электромонтажные работы
5,3,106,1,0,"{""3"": 1}",малярные работы
6,21,102,1,0,"{""21"": 0}",штукатурка стен
7,30,105,0,1,"{""30"": 0}","монтаж дверей, света и сантехники"
8,38,106,0,1,"{""38"": 1, ""187"": 1}",Отделка под покраску и обои
9,43,106,0,1,"{""43"": 1, ""47"": 1, ""98"": 1, ""116"": 0}",Штукатурные и шпаклевочные работы под покраску или обои


Тексты и ID тестов совпадают. Пересечений train с gold и публичным тестом после нормализации нет.


In [3]:
dev_ids = {row['item_id'] for row in old_train if row['origin'] == 'real'
           and int(digest(key(row['description']))[:8], 16) % 5 == 0}
# Дополнительно исключаем совпадения с hard и диагностическими примерами на время замера.
prep_stress = [r for r in new_stress if r['subslice'] == 'prep_only']
eval_rows = new_gold + new_hard + prep_stress
reserved = {key(row['description']) for row in eval_rows}
reserved_norm = {normalize(row['description']) for row in eval_rows} - {''}
overlaps = {row['source_item_id'] for row in old_train if row['origin'] == 'real'
            and (key(row['description']) in reserved or normalize(row['description']) in reserved_norm)}
def eligible(row):
    return (row['source_item_id'] not in dev_ids | overlaps
            and key(row['description']) not in reserved
            and normalize(row['description']) not in reserved_norm)

old_fit, new_fit = [[row for row in rows if eligible(row)] for rows in (old_train, new_train)]
assert [row['item_id'] for row in old_fit] == [row['item_id'] for row in new_fit]
old_real, new_real = [[row for row in rows if row['origin'] == 'real'] for rows in (old_fit, new_fit)]
raw_control = [{**r, 'found_microcategories': raw[r['source_line']]['found_microcategories']} for r in old_real]
candidates = {'Исходные метки на тех же реальных строках': raw_control,
              'Прошлый train, реальные': old_real, 'Train по справочнику, реальные': new_real,
              'Прошлый train с синтетикой': old_fit, 'Train по справочнику с синтетикой': new_fit}
sections = {'Прежний gold': [canonical(r, 101) for r in old_gold],
            'Gold по справочнику': [canonical(r, 101) for r in new_gold],
            'Hard по справочнику': [canonical(r, 101) for r in new_hard],
            'Подготовка в stress': [canonical(r, 101) for r in prep_stress]}
report = []
for name, rows in candidates.items():
    gold_predictions = list(baseline([canonical(r, 101) for r in rows],
                                     sections['Gold по справочнику'] + sections['Hard по справочнику']
                                     + sections['Подготовка в stress']))
    start = 0
    for label, samples in sections.items():
        offset = 0 if label == 'Прежний gold' else start
        predictions = gold_predictions[offset:offset + len(samples)]
        score = metrics(samples, predictions)
        fp = sum(106 in p['labels'] and 106 not in t['labels'] for t, p in zip(samples, predictions))
        fn = sum(106 in t['labels'] and 106 not in p['labels'] for t, p in zip(samples, predictions))
        report.append({'train': name, 'fit': len(rows), 'test': label,
                       'micro-F1': round(score['micro_f1'], 4), 'exact': round(score['exact_match'], 4),
                       'FP 106': fp, 'FN 106': fn})
        if label != 'Прежний gold':
            start += len(samples)
benchmark_table = pd.DataFrame(report)
display(benchmark_table)
print(f'Dev: {len(dev_ids)} реальных строк. Fit использует общие ID, тексты и порог 0.5 для каждой пары.')


,train,fit,test,micro-F1,exact,FP 106,FN 106
0,Исходные метки на тех же реальных строках,3404,Прежний gold,0.8715,0.5750,6,3
1,Исходные метки на тех же реальных строках,3404,Gold по справочнику,0.8725,0.5750,3,2
2,Исходные метки на тех же реальных строках,3404,Hard по справочнику,0.5315,0.2800,3,9
3,Исходные метки на тех же реальных строках,3404,Подготовка в stress,0.7812,0.4375,0,6
4,"Прошлый train, реальные",3404,Прежний gold,0.9049,0.7125,6,2
5,"Прошлый train, реальные",3404,Gold по справочнику,0.9006,0.6875,4,2
6,"Прошлый train, реальные",3404,Hard по справочнику,0.5556,0.3000,3,9
7,"Прошлый train, реальные",3404,Подготовка в stress,0.7333,0.3125,0,7
8,"Train по справочнику, реальные",3404,Прежний gold,0.9049,0.7125,6,2
9,"Train по справочнику, реальные",3404,Gold по справочнику,0.9006,0.6875,4,2


Dev: 810 реальных строк. Fit использует общие ID, тексты и порог 0.5 для каждой пары.
